# Ljungqvist & Sargent (2018), Chapter 6 --- Search and Unemployment

Companion notebook to `ls_solutions_ch06.pdf`. One section per end-of-chapter exercise (6.1--6.27,
pp. 204--222). Every numerical answer is computed one way and re-derived by an independent route
(the reservation-wage equation vs. value-function iteration on the whole function, policy evaluation
of cutoff strategies, exhaustive policy enumeration, or Monte Carlo simulation), with an `assert`
that aborts on mismatch. Matlab requests (6.18, 6.25) are done in Python.

Notation follows the book: offers $w\sim F$, compensation $c$, discount $\beta$, reservation wage
$\bar w$ (called `wbar`), $Q$ the value of rejecting. Where the book gives no numbers we choose
standard ones and state them in each section. Figures go to `fig/ls06_*.pdf` with TrueType fonts.

In [1]:
import numpy as np
from scipy import optimize
import matplotlib.pyplot as plt
from pathlib import Path
import logging
logging.getLogger("fontTools").setLevel(logging.ERROR)   # silence font-subsetting chatter

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 9, "font.family": "serif",
                     "mathtext.fontset": "cm", "legend.fontsize": 7.5, "axes.titlesize": 9})
FIG = Path("fig"); FIG.mkdir(exist_ok=True)
rng = np.random.default_rng(20180606)

def save_pdf(fig, name):
    """Write fig to fig/ls06_<name>.pdf (TrueType fonts, never Type 3)."""
    fig.savefig(FIG / f"ls06_{name}.pdf", bbox_inches="tight")

def unif_grid(n, lo=0.0, hi=1.0):
    """Discrete uniform offer distribution: n cell midpoints of [lo, hi], equal weights
    (so F(lo) = 0 and F(hi) = 1, as the book assumes)."""
    return lo + (np.arange(n) + .5) * (hi - lo) / n, np.full(n, 1 / n)

def G(x, w, p):
    """Option value integral  int_x (w' - x) dF(w')  for a discrete distribution (w, p)."""
    return np.sum(p * np.maximum(w - x, 0.0))

def resw(c, beta, w, p, scale=1.0):
    """Reservation wage solving  wbar - c = scale * beta/(1-beta) * G(wbar)  ((6.3.3) when scale=1).
    The left side minus the right side is strictly increasing, so the root is unique (brentq)."""
    f = lambda x: x - c - scale * beta / (1 - beta) * G(x, w, p)
    return optimize.brentq(f, min(c, w.min()) - 1, max(c, w.max()) + 1, xtol=1e-14)

def mccall_vfi(c, beta, w, p, tol=1e-12):
    """Iterate v <- max{w/(1-beta), c + beta E v} on the whole function; return v and the implied
    reservation wage (1-beta)(c + beta E v)."""
    v = w / (1 - beta)
    for _ in range(100000):
        vn = np.maximum(w / (1 - beta), c + beta * p @ v)
        if np.max(np.abs(vn - v)) < tol:
            return vn, (1 - beta) * (c + beta * p @ vn)
        v = vn
    raise RuntimeError("VFI did not converge")

def cutoff_value(x, c, beta, w, p):
    """Policy evaluation: value before drawing of the stationary rule 'accept iff w >= x'.
    U = [sum_{w>=x} p w/(1-beta) + P(w<x) c] / (1 - beta P(w<x))."""
    acc = w >= x
    rej = p[~acc].sum()
    return (p[acc] @ w[acc] / (1 - beta) + rej * c) / (1 - beta * rej)

def max_dist(p, n):
    """Probabilities of the best of n i.i.d. draws from the discrete distribution p (n may be real)."""
    Fc = np.clip(np.cumsum(p), 0, 1) ** n
    return np.diff(np.concatenate([[0.0], Fc]))

## Exercise 6.1 --- being unemployed with a chance of an offer

$v(w)=\max\{w/(1-\beta),\ c+\beta[\phi v(0)+(1-\phi)\int v\,dF]\}$, so
$\bar w-c=\frac{\beta(1-\phi)}{1-\beta}\int_{\bar w}(w'-\bar w)dF$.
Parameters: $F$ uniform on $[0,10]$ (200-point grid), $c=1$, $\beta=.95$, $\phi\in\{0,.3,.6\}$.
Checks: equation vs. VFI with an explicit zero offer; mean duration $1/[(1-\phi)(1-F(\bar w))]$ vs. simulation.

In [2]:
w10, p10 = unif_grid(200, 0, 10)
c61, b61 = 1.0, .95
res61 = {}
for phi in (0.0, .3, .6):
    wb = resw(c61, b61, w10, p10, scale=1 - phi)
    # VFI on the augmented offer set {0} U grid, P(0) = phi
    wa, pa = np.concatenate([[0.], w10]), np.concatenate([[phi], (1 - phi) * p10])
    _, wb_vfi = mccall_vfi(c61, b61, wa, pa)
    assert abs(wb - wb_vfi) < 1e-9
    haz = (1 - phi) * p10[w10 >= wb].sum()
    # simulate durations
    T = np.zeros(20000)
    for i in range(T.size):
        t = 1
        while not (rng.random() > phi and w10[rng.integers(200)] >= wb):
            t += 1
        T[i] = t
    se = T.std() / np.sqrt(T.size)
    assert abs(T.mean() - 1 / haz) < 4 * se
    res61[phi] = wb
    print(f"phi={phi}: wbar={wb:.6f} (VFI {wb_vfi:.6f}); mean duration {1/haz:.4f}, simulated {T.mean():.4f}")
assert res61[0.] > res61[.3] > res61[.6]

phi=0.0: wbar=7.403704 (VFI 7.403704); mean duration 3.8462, simulated 3.8224


phi=0.3: wbar=6.996994 (VFI 6.996994); mean duration 4.7619, simulated 4.7637
phi=0.6: wbar=6.274351 (VFI 6.274351); mean duration 6.6667, simulated 6.7247


## Exercise 6.2 --- two offers per period

The best of two draws has c.d.f. $F^2$: $\bar w_2-c=\frac{\beta}{1-\beta}\int_{\bar w_2}(w'-\bar w_2)dF^2$.
Checks: VFI on the *pair* of offers $(w_1,w_2)$ (never using $F^2$); policy evaluation of all cutoffs;
$\bar w_2>\bar w_1$ for a range of $c$.

In [3]:
w1, p1 = unif_grid(120, 0, 10)
b62 = .95
for c in (0., 1., 3., 6.):
    wb1 = resw(c, b62, w1, p1)
    wb2 = resw(c, b62, w1, max_dist(p1, 2))
    # VFI on pairs
    M = np.maximum.outer(w1, w1) / (1 - b62); P2 = np.outer(p1, p1)
    v = M.copy()
    for _ in range(5000):
        vn = np.maximum(M, c + b62 * np.sum(P2 * v))
        if np.max(np.abs(vn - v)) < 1e-11: break
        v = vn
    wb2_pair = (1 - b62) * (c + b62 * np.sum(P2 * vn))
    assert abs(wb2 - wb2_pair) < 1e-8
    # best cutoff among grid points by policy evaluation
    vals = [cutoff_value(x, c, b62, w1, max_dist(p1, 2)) for x in w1]
    assert w1[int(np.argmax(vals))] == w1[w1 >= wb2 - 1e-12][0]
    assert wb2 > wb1
    print(f"c={c}: one offer wbar={wb1:.5f}, two offers wbar={wb2:.5f} (pair-VFI {wb2_pair:.5f})")
# the gap h_2(w) - h_1(w) = beta/(1-beta) int_w^B [F - F^2] dw' >= 0
xs = np.linspace(0, 10, 11)
h1 = np.array([G(x, w1, p1) for x in xs]); h2 = np.array([G(x, w1, max_dist(p1, 2)) for x in xs])
assert np.all(h2[:-1] > h1[:-1])

c=0.0: one offer wbar=7.23946, two offers wbar=7.88660 (pair-VFI 7.88660)
c=1.0: one offer wbar=7.40368, two offers wbar=8.01185 (pair-VFI 8.01185)
c=3.0: one offer wbar=7.76126, two offers wbar=8.28239 (pair-VFI 8.28239)
c=6.0: one offer wbar=8.40792, two offers wbar=8.76738 (pair-VFI 8.76738)


## Exercise 6.3 --- a random number of offers per period

Best offer has c.d.f. $\sum_n\pi_nF^n$. Example: $\pi=(.2,.5,.3)$ for $n=1,2,3$, $F$ uniform on
$[0,10]$, $c=1$, $\beta=.95$. Check: equation vs. VFI on the triple $(n, w_1..w_n)$ simulated as the
exact mixture, and vs. policy evaluation of cutoffs. Also: $\bar w$ lies between the $n=1$ and $n=3$ values.

In [4]:
pi63 = np.array([.2, .5, .3]); c63, b63 = 1., .95
H63 = sum(pi63[k] * max_dist(p1, k + 1) for k in range(3))
wb63 = resw(c63, b63, w1, H63)
# independent: VFI where the state is (n, best offer) and E v is taken as sum_n pi_n E_{F^n}
V = np.tile(w1 / (1 - b63), (3, 1))
for _ in range(5000):
    EV = sum(pi63[k] * max_dist(p1, k + 1) @ V[k] for k in range(3))
    Vn = np.maximum(w1 / (1 - b63), c63 + b63 * EV)[None, :].repeat(3, 0)
    if np.max(np.abs(Vn - V)) < 1e-11: break
    V = Vn
wb63_vfi = (1 - b63) * (c63 + b63 * EV)
assert abs(wb63 - wb63_vfi) < 1e-8
vals = [cutoff_value(x, c63, b63, w1, H63) for x in w1]
assert w1[int(np.argmax(vals))] == w1[w1 >= wb63][0]
lo, hi = resw(c63, b63, w1, p1), resw(c63, b63, w1, max_dist(p1, 3))
assert lo < wb63 < hi
print(f"random n: wbar={wb63:.5f} (VFI {wb63_vfi:.5f}); n=1 gives {lo:.5f}, n=3 gives {hi:.5f}")

random n: wbar=8.03679 (VFI 8.03679); n=1 gives 7.40368, n=3 gives 8.30439


## Exercise 6.4 --- Markov number of offers

$v(w,n)=\max\{w/(1-\beta),\ Q(n)\}$, $Q(n)=c+\beta\sum_m\pi_{mn}\int v(w',m)\,dF^m(w')$, so $\bar w(n)=(1-\beta)Q(n)$.
Example: $N=3$, persistent chain (rows = current $n$): diag .8. Check: fixed point in the 3-vector $Q$
(reservation-wage equations) vs. VFI on the full $(w,n)$ grid; and $\bar w(n)$ increasing in $n$ here.

In [5]:
Pi = np.array([[.8, .15, .05], [.1, .8, .1], [.05, .15, .8]])   # Pi[n, m] = Prob(m next | n now) = book's pi_{mn}
c64, b64 = 1., .95
Hm = [max_dist(p1, m + 1) for m in range(3)]
Q = np.zeros(3)
for _ in range(10000):   # Q(n) = c + beta sum_m pi_mn [Q(m) + G_m(wbar_m)/(1-beta)]
    wbm = (1 - b64) * Q
    Qn = c64 + b64 * Pi @ np.array([Q[m] + G(wbm[m], w1, Hm[m]) / (1 - b64) for m in range(3)])
    if np.max(np.abs(Qn - Q)) < 1e-12: break
    Q = Qn
wb64 = (1 - b64) * Qn
V = np.tile(w1 / (1 - b64), (3, 1))
for _ in range(10000):
    EV = np.array([Hm[m] @ V[m] for m in range(3)])
    Vn = np.maximum(w1[None, :] / (1 - b64), (c64 + b64 * Pi @ EV)[:, None])
    if np.max(np.abs(Vn - V)) < 1e-11: break
    V = Vn
wb64_vfi = (1 - b64) * (c64 + b64 * Pi @ np.array([Hm[m] @ Vn[m] for m in range(3)]))
assert np.allclose(wb64, wb64_vfi, atol=1e-8)
# the policy from VFI is a cutoff in w for each n
for n in range(3):
    acc = Vn[n] <= w1 / (1 - b64) + 1e-12
    assert np.all(np.diff(acc.astype(int)) >= 0)
assert np.all(np.diff(wb64) > 0)
print("wbar(n) for n=1,2,3:", wb64, "(VFI:", wb64_vfi, ")")

wbar(n) for n=1,2,3:

 [7.707647 7.978346 8.163679] (VFI: [7.707647 7.978346 8.163679] )


## Exercise 6.5 --- choosing the number of offers

$Q=\max_{n\ge1}\{-k(n)+\int\max\{w/(1-\beta),\,c+\beta Q\}\,dF^n(w)\}$. Example: $k(n)=2n$,
$F$ uniform on $[0,10]$, $c=1$, $\beta=.95$. Check: iterate the Bellman equation for $Q$ vs. evaluate every
stationary rule (n, cutoff x) exactly and take the best.

In [6]:
c65, b65, kap = 1., .95, 2.
nmax = 15
Hn = [max_dist(p1, n) for n in range(1, nmax + 1)]
Q = 0.
for _ in range(100000):
    obj = np.array([-kap * n + Hn[n - 1] @ np.maximum(w1 / (1 - b65), c65 + b65 * Q) for n in range(1, nmax + 1)])
    Qn = obj.max()
    if abs(Qn - Q) < 1e-12: break
    Q = Qn
n_star = int(np.argmax(obj)) + 1; wb65 = (1 - b65) * (c65 + b65 * Qn)
best = -np.inf
for n in range(1, nmax + 1):
    H = Hn[n - 1]
    for x in w1:
        acc = w1 >= x; rej = H[~acc].sum()
        U = (-kap * n + H[acc] @ w1[acc] / (1 - b65) + rej * c65) / (1 - b65 * rej)
        if U > best: best, arg = U, (n, x)
assert abs(best - Qn) < 1e-8 and arg[0] == n_star and arg[1] == w1[w1 >= wb65][0]
print(f"optimal n = {n_star}, Q = {Qn:.6f} (enumeration {best:.6f}), wbar = {wb65:.5f}")

optimal n = 6, Q = 164.802326 (enumeration 164.802326), wbar = 7.87811

## Exercise 6.6 --- Mortensen externality

$\theta\sim U[0,1]$; $n=f(c_1+c_2)=1+4\sqrt{c_1+c_2}$ (treated as a real number: the best of $n$ draws has
c.d.f. $\theta^n$); $\beta=.9$. With $R$ the value of staying unmatched next period and $r=\min\{(1-\beta)R,1\}$,
$M(n,R)=E_{F^n}\max\{\theta/(1-\beta),R\}=Rr^n+\frac{n(1-r^{n+1})}{(n+1)(1-\beta)}$.
Nash (symmetric): $Q=-c+M(f(2c),\beta Q)$ with $c$ maximising $-c_i+M(f(c_i+c),\beta Q)$.
Planner with weight $\lambda$: $Q=\max_C\{-\min(\lambda,1-\lambda)C+M(f(C),\beta Q)\}$.
Checks: $M$ vs. quadrature; Nash $c$ is a best response by direct maximisation; planner total $C$ exceeds Nash.

In [7]:
b66 = .9
f66 = lambda C: 1 + 4 * np.sqrt(C)
def M66(n, R):
    r = min(max((1 - b66) * R, 0.), 1.)
    return R * r**n + n * (1 - r**(n + 1)) / ((n + 1) * (1 - b66))
from scipy import integrate
for n, R in [(1.7, 3.), (4.2, 6.5), (2., 20.)]:
    quad = integrate.quad(lambda t: max(t / (1 - b66), R) * n * t**(n - 1), 0, 1, points=[min((1 - b66) * R, 1)])[0]
    assert abs(quad - M66(n, R)) < 1e-9
def best_c(cj, Q, wt=1.0):
    """argmax_{ci>=0} -wt*ci + M(f(ci+cj), beta Q) by bounded scalar search."""
    r = optimize.minimize_scalar(lambda ci: wt * ci - M66(f66(ci + cj), b66 * Q), bounds=(0, 5), method="bounded",
                                 options={"xatol": 1e-12})
    return r.x, -r.fun
# Nash: iterate on (c, Q)
c, Q = .1, 0.
for _ in range(5000):
    cn, val = best_c(c, Q)
    Qn = val
    if abs(cn - c) < 1e-10 and abs(Qn - Q) < 1e-10: break
    c, Q = cn, Qn
cN, QN = cn, Qn
# FOC route: 1 = f'(2c) dM/dn
dM = lambda n, R, h=1e-6: (M66(n + h, R) - M66(n - h, R)) / (2 * h)
fp = lambda C: 2 / np.sqrt(C)
assert abs(fp(2 * cN) * dM(f66(2 * cN), b66 * QN) - 1) < 1e-4
# Planner for lambda = .5 and .3
res66 = {}
for lam in (.5, .3):
    wt = min(lam, 1 - lam)
    Q = 0.
    for _ in range(5000):
        C, Qn = best_c(0., Q, wt)
        if abs(Qn - Q) < 1e-11: break
        Q = Qn
    assert abs(fp(C) * dM(f66(C), b66 * Qn) - wt) < 1e-4
    res66[lam] = (C, Qn)
    print(f"planner lambda={lam}: total search C={C:.5f}, n={f66(C):.4f}, planner value={Qn:.5f}")
print(f"Nash: c1=c2={cN:.5f}, total C={2*cN:.5f}, n={f66(2*cN):.4f}, Q_i={QN:.5f}, reservation theta={(1-b66)*b66*QN:.5f}")
assert res66[.5][0] > 2 * cN and res66[.3][0] > res66[.5][0]
# utilitarian value per agent: planner (lambda=.5) beats Nash
assert res66[.5][1] > QN

planner lambda=0.5: total search C=0.54950, n=3.9651, planner value=8.14116
planner lambda=0.3: total search C=0.94528, n=4.8890, planner value=8.32953
Nash: c1=c2=0.11112, total C=0.22225, n=2.8857, Q_i=8.05157, reservation theta=0.72464


## Exercise 6.7 --- variable labor supply

$u(y,l)=2\sqrt y-1/l$ ($\gamma=.5$, $\eta=2$ in $y^{1-\gamma}/(1-\gamma)+l^{1-\eta}/(1-\eta)$); unemployed: $y=c=1$,
$l=1$. Employed: $u^*(w)=\max_l u(w(1-l),l)$ every period. $F$ uniform on $[0,10]$, $\beta=.95$.
Checks: $l^*(w)$ from the FOC $1/l^2=\sqrt w/\sqrt{1-l}$ vs. numerical maximisation; reservation equation
$u^*(\bar w)-u(c,1)=\frac{\beta}{1-\beta}\int_{\bar w}[u^*(w)-u^*(\bar w)]dF$ vs. VFI.

In [8]:
b67, c67 = .95, 1.
u67 = lambda y, l: 2 * np.sqrt(y) - 1 / l
def lstar(w):
    return optimize.brentq(lambda l: 1 / l**2 - np.sqrt(w) / np.sqrt(1 - l), 1e-9, 1 - 1e-15, xtol=1e-15)
wg, pg = unif_grid(200, 0, 10)
L = np.array([lstar(w) for w in wg])
Lnum = np.array([optimize.minimize_scalar(lambda l: -u67(w * (1 - l), l), bounds=(1e-6, 1 - 1e-9), method="bounded",
                                          options={"xatol": 1e-12}).x for w in wg])
assert np.allclose(L, Lnum, atol=1e-5)
ustar = u67(wg * (1 - L), L)
assert np.all(np.diff(ustar) > 0)
uu = u67(c67, 1.)
wb67 = optimize.brentq(lambda x: np.interp(x, wg, ustar) - uu
                       - b67 / (1 - b67) * np.sum(pg * np.maximum(ustar - np.interp(x, wg, ustar), 0)), wg[0], wg[-1])
v = ustar / (1 - b67)
for _ in range(100000):
    vn = np.maximum(ustar / (1 - b67), uu + b67 * pg @ v)
    if np.max(np.abs(vn - v)) < 1e-11: break
    v = vn
acc = ustar / (1 - b67) >= uu + b67 * pg @ vn
assert np.all(np.diff(acc.astype(int)) >= 0)          # accept set is an upper interval: reservation wage
assert wg[acc][0] >= wb67 > wg[~acc][-1]
print(f"wbar = {wb67:.5f}; hours 1-l* at w=wbar: {1-lstar(wb67):.4f}, at w=10: {1-lstar(9.99):.4f}")

wbar = 7.93482; hours 1-l* at w=wbar: 0.4992, at w=10: 0.5219


## Exercise 6.8 --- wage growth and the reservation wage

Accepting $w$ is worth $w/(1-\beta\phi)$; the reservation wage solves $(1-\beta)\bar w=(1-\beta\phi)c+\beta\int_{\bar w}(w'-\bar w)dF$.
$F$ uniform on $[0,10]$, $\beta=.95$, $\phi\in\{1,1.02,1.04\}$, $c\in\{1,0\}$. Check vs. VFI.

In [9]:
b68 = .95
res68 = {}
for c in (1., 0.):
    for phi in (1., 1.02, 1.04):
        wb = optimize.brentq(lambda x: (1 - b68) * x - (1 - b68 * phi) * c - b68 * G(x, w1, p1), -1, 20, xtol=1e-14)
        v = w1 / (1 - b68 * phi)
        for _ in range(100000):
            vn = np.maximum(w1 / (1 - b68 * phi), c + b68 * p1 @ v)
            if np.max(np.abs(vn - v)) < 1e-11: break
            v = vn
        wb_vfi = (1 - b68 * phi) * (c + b68 * p1 @ vn)
        assert abs(wb - wb_vfi) < 1e-8
        res68[c, phi] = wb
        print(f"c={c}, phi={phi}: wbar={wb:.6f}")
assert res68[1., 1.] > res68[1., 1.02] > res68[1., 1.04]
assert abs(res68[0., 1.] - res68[0., 1.04]) < 1e-10       # with c = 0 growth does not matter

c=1.0, phi=1.0: wbar=7.403679
c=1.0, phi=1.02: wbar=7.340293
c=1.0, phi=1.04: wbar=7.278012
c=0.0, phi=1.0: wbar=7.239458
c=0.0, phi=1.02: wbar=7.239458
c=0.0, phi=1.04: wbar=7.239458


## Exercise 6.9 --- search with a finite horizon (two periods)

$\bar w_2=c$, $\bar w_1=[c+\beta E\max(w,c)]/(1+\beta)\ge c$. $F$ uniform on $[0,1]$, $c=.3$, $\beta=.95$.
Check: brute-force search over all pairs of cutoffs $(x_1,x_2)$ on the grid.

In [10]:
wu, pu = unif_grid(200, 0, 1)
b69, c69 = .95, .3
wb2 = c69
wb1 = (c69 + b69 * pu @ np.maximum(wu, c69)) / (1 + b69)
def payoff(x1, x2):
    a1 = wu >= x1
    per2 = pu @ np.where(wu >= x2, wu, c69)
    return pu[a1] @ ((1 + b69) * wu[a1]) + pu[~a1].sum() * (c69 + b69 * per2)
X = wu
vals = np.array([[payoff(a, b) for b in X] for a in X])
i, j = np.unravel_index(np.argmax(vals), vals.shape)
assert X[i] == X[X >= wb1][0] and X[j] == X[X >= wb2][0]
assert wb1 > wb2
print(f"wbar_1 = {wb1:.5f} > wbar_2 = c = {wb2}")

wbar_1 = 0.41936 > wbar_2 = c = 0.3


## Exercise 6.10 --- finite horizon and mean-preserving spread

$F_2$: triangular on $[0,1]$; $F_1$: uniform on $[0,1]$ (an MPS of $F_2$: same mean, (iii) holds).
$A_t=\sum_{s=0}^{T-t}\beta^s$, $Q_T=c$, $Q_t=c+\beta E\max\{A_{t+1}w,Q_{t+1}\}$, $\bar w_t=Q_t/A_t$.
$T=6$, $\beta=.95$, $c\in\{0,.2\}$. Check: backward induction on the value *functions* $V_t(w)$.

In [11]:
n610 = 400
wt, pF1 = unif_grid(n610, 0, 1)
pF2 = 1 - np.abs(2 * wt - 1); pF2 /= pF2.sum()
assert abs(pF1 @ wt - pF2 @ wt) < 1e-12                                  # (i) equal means
iii = np.cumsum(np.cumsum(pF1) - np.cumsum(pF2)) / n610                 # int_0^y (F1 - F2)
assert np.all(iii >= -1e-12)
T610, b610 = 6, .95
Aann = lambda t: sum(b610**s for s in range(T610 - t + 1))
def resw_T(p, c):
    Q = {T610: c}
    for t in range(T610 - 1, -1, -1):
        Q[t] = c + b610 * p @ np.maximum(Aann(t + 1) * wt, Q[t + 1])
    return np.array([Q[t] / Aann(t) for t in range(T610 + 1)])
def resw_T_functions(p, c):
    V = np.maximum(wt, c)       # V_T
    out = [c]
    for t in range(T610 - 1, -1, -1):
        cont = c + b610 * p @ V
        out.append(cont / Aann(t))
        V = np.maximum(Aann(t) * wt, cont)
    return np.array(out[::-1])
res610 = {}
for c in (0., .2):
    r1, r2 = resw_T(pF1, c), resw_T(pF2, c)
    assert np.allclose(r1, resw_T_functions(pF1, c)) and np.allclose(r2, resw_T_functions(pF2, c))
    res610[c] = (r1, r2)
    print(f"c={c}\n  F1 (riskier):", r1, "\n  F2          :", r2)
r1, r2 = res610[0.]
assert abs(r1[T610] - r2[T610]) < 1e-12 and abs(r1[T610 - 1] - r2[T610 - 1]) < 1e-12       # (a)
assert np.all(r1[:T610 - 1] > r2[:T610 - 1])                                             # (b)
r1, r2 = res610[.2]
assert abs(r1[T610] - r2[T610]) < 1e-12 and r1[T610 - 1] > r2[T610 - 1]                     # (c)

c=0.0
  F1 (riskier): [0.517034 0.489404 0.454463 0.408439 0.343982 0.24359  0.      ] 
  F2          : [0.466239 0.445308 0.418637 0.38288  0.330973 0.24359  0.      ]
c=0.2
  F1 (riskier): [0.57917  0.555883 0.526655 0.488549 0.435958 0.355897 0.2     ] 
  F2          : [0.521473 0.503987 0.482124 0.453488 0.413194 0.348752 0.2     ]


## Exercise 6.11 --- Pissarides: taxation and variable search intensity

$\pi(e)=1-e^{-2e}$, $F$ uniform on $[0,5]$, $\beta=.95$, $\alpha=.1$, $z=.5$, $T(w)=t(w-a)$.
With $D=1-\beta(1-\alpha)$ and $G(x)=\int_x(w-x)dF=(5-x)^2/10$ the solution solves
$(1-t)\bar w=1+z-ta+\beta(1-\alpha)[\pi(e)(1-t)G(\bar w)/D-e]$ and $\pi'(e)(1-t)G(\bar w)/D=1$.
Checks: (1) VFI on the literal Bellman equation (grids for $w$ and $e$, $W(w)$ iterated as an array);
(2) finite-difference comparative statics at $a\in\{.2,2.5\}$; the sign of $\partial\bar w/\partial t$ flips at
$a=1+z-\beta(1-\alpha)e$.

In [12]:
b611, al611, z611, B611 = .95, .1, .5, 5.
D611 = 1 - b611 * (1 - al611)
pie = lambda e: 1 - np.exp(-2 * e)
Gu = lambda x: (B611 - x) ** 2 / (2 * B611)
def e_of(wb, t):
    """Effort from the FOC 2 exp(-2e) (1-t) G(wb)/D = 1 (corner e = 0 if the marginal benefit is < 1 at 0)."""
    k = 2 * (1 - t) * Gu(wb) / D611
    return .5 * np.log(k) if k > 1 else 0.
def solve611(a, t):
    def resid(wb):
        e = e_of(wb, t)
        return (1 - t) * wb - (1 + z611 - t * a) - b611 * (1 - al611) * (pie(e) * (1 - t) * Gu(wb) / D611 - e)
    wb = optimize.brentq(resid, 0., B611, xtol=1e-14)
    return wb, e_of(wb, t)
a0, t0 = 1., .3
wb611, e611 = solve611(a0, t0)
# (1) literal VFI
wv, pv = unif_grid(1000, 0, B611)
eg = np.linspace(0, 3, 1201)
U, W = 0., np.zeros_like(wv)
net = wv - t0 * (wv - a0)
for _ in range(20000):
    rej = 1 - eg + z611 + b611 * U                                  # income 1-e+z today, U tomorrow
    offer = np.maximum(W[None, :] - eg[:, None], rej[:, None]) @ pv  # accept: W(w) - e ; reject: rej
    vals = (1 - pie(eg)) * rej + pie(eg) * offer
    Un = vals.max(); e_vfi = eg[vals.argmax()]
    Wn = net + b611 * ((1 - al611) * W + al611 * U)
    if abs(Un - U) < 1e-10 and np.max(np.abs(Wn - W)) < 1e-10: break
    U, W = Un, Wn
wb_vfi = wv[W >= 1 + z611 + b611 * U][0]
assert abs(wb_vfi - wb611) < 6e-3 and abs(e_vfi - e611) < 3e-3
print(f"a={a0}, t={t0}: wbar={wb611:.5f} (VFI {wb_vfi:.5f}), e={e611:.5f} (VFI {e_vfi:.4f})")
# (2) comparative statics
h = 1e-5
dwa = (solve611(a0 + h, t0)[0] - solve611(a0 - h, t0)[0]) / (2 * h)
dea = (solve611(a0 + h, t0)[1] - solve611(a0 - h, t0)[1]) / (2 * h)
assert dwa < 0 and dea > 0
res611 = {}
for a in (.2, 2.5):
    wb, e = solve611(a, t0)
    dwt = (solve611(a, t0 + h)[0] - solve611(a, t0 - h)[0]) / (2 * h)
    det = (solve611(a, t0 + h)[1] - solve611(a, t0 - h)[1]) / (2 * h)
    thr = 1 + z611 - b611 * (1 - al611) * e
    assert np.sign(dwt) == np.sign(thr - a)                           # dwbar/dt > 0  iff  a < 1+z-beta(1-alpha)e
    Phi_w = (1 - t0) * (1 + b611 * (1 - al611) * pie(e) * (1 - wb / B611) / D611)
    assert abs(dwt - (-(a - thr) / (1 - t0)) / Phi_w) < 1e-5         # analytic dwbar/dt
    res611[a] = (wb, e, dwt, det, thr)
    print(f"a={a}: wbar={wb:.4f}, e={e:.4f}, dwbar/dt={dwt:.4f}, de/dt={det:.4f}, threshold 1+z-beta(1-alpha)e={thr:.4f}")
print(f"dwbar/da={dwa:.4f}, de/da={dea:.4f}")
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.6))
agrid = np.linspace(0, 3, 61); tgrid = np.linspace(.05, .6, 56)
sa = np.array([solve611(a, t0) for a in agrid])
ax[0].plot(agrid, sa[:, 0], label=r"$\bar w$"); ax[0].plot(agrid, 5 * sa[:, 1], "--", label=r"$5e$")
ax[0].set_xlabel(r"$a$ (with $t=.3$)"); ax[0].legend(); ax[0].set_title("Raising the exemption $a$")
for a, ls in ((.2, "-"), (2.5, ":")):
    st = np.array([solve611(a, t) for t in tgrid])
    ax[1].plot(tgrid, st[:, 0], ls, label=rf"$\bar w$, $a={a}$")
ax[1].set_xlabel(r"$t$"); ax[1].legend(); ax[1].set_title("Raising the tax rate $t$")
fig.tight_layout(); save_pdf(fig, "pissarides"); plt.close(fig)

a=1.0, t=0.3: wbar=2.87227 (VFI 2.87250), e=0.73751 (VFI 0.7375)
a=0.2: wbar=2.9927, e=0.6793, dwbar/dt=0.5321, de/dt=-0.9793, threshold 1+z-beta(1-alpha)e=0.9192
a=2.5: wbar=2.6639, e=0.8310, dwbar/dt=-1.0800, de/dt=-0.2520, threshold 1+z-beta(1-alpha)e=0.7895
dwbar/da=-0.1460, de/da=0.0686


## Exercise 6.12 --- search and financial income

Subtract $\phi\epsilon_t$ (received whatever the worker does) from every period's income: the problem becomes
McCall with i.i.d. compensation $\tilde c_t=c+(1-\phi)\epsilon_t$, so $\bar w(\epsilon)=(1-\beta)[c+(1-\phi)\epsilon+\beta U]$,
$U=E\max\{w/(1-\beta),\,c+(1-\phi)\epsilon+\beta U\}$. $w\sim U[0,10]$, $\epsilon\sim U[0,4]$, $c=1$, $\phi=.5$, $\beta=.95$.
Check: VFI on the *original* problem on the $(w,\epsilon)$ grid.

In [13]:
b612, c612, ph612 = .95, 1., .5
ee, pe = unif_grid(40, 0, 4)
def solve612(ee):
    mu = pe @ ee
    Ufix = optimize.brentq(lambda U: U - pe @ np.array([p1 @ np.maximum(w1 / (1 - b612), c612 + (1 - ph612) * e + b612 * U)
                                                       for e in ee]), 0, 1e4, xtol=1e-12)
    wbar = (1 - b612) * (c612 + (1 - ph612) * ee + b612 * Ufix)
    # original problem: accept -> w/(1-b) + phi e + b phi mu/(1-b); reject -> c + e + b E v
    acc = w1[:, None] / (1 - b612) + ph612 * ee[None, :] + b612 * ph612 * mu / (1 - b612)
    v = acc.copy()
    for _ in range(100000):
        Ev = p1 @ v @ pe
        vn = np.maximum(acc, c612 + ee[None, :] + b612 * Ev)
        if np.max(np.abs(vn - v)) < 1e-10: break
        v = vn
    Ev = p1 @ vn @ pe
    wbar_vfi = (1 - b612) * (c612 + ee + b612 * Ev - ph612 * ee - b612 * ph612 * mu / (1 - b612))
    return wbar, wbar_vfi
wb612, wb612v = solve612(ee)
assert np.allclose(wb612, wb612v, atol=1e-8)
assert np.allclose(np.diff(wb612) / np.diff(ee), (1 - b612) * (1 - ph612))
prob = np.mean([p1[w1 >= c612 + (1 - ph612) * e].sum() for e in ee])
assert 0 < prob < 1
wb612s, _ = solve612(ee + 1.)          # higher mean financial income
assert np.all(wb612s > wb612)
print("wbar(eps) at eps = 0.05, 1.95, 3.95:", wb612[[0, 19, 39]], "; slope", (1 - b612) * (1 - ph612))
print("after +1 shift in eps:", wb612s[[0, 19, 39]])

wbar(eps) at eps = 0.05, 1.95, 3.95: [7.528342 7.575842 7.625842] ; slope 0.025000000000000022
after +1 shift in eps: [7.61896 7.66646 7.71646]


## Exercise 6.13 --- search and asset accumulation (numerical illustration)

One value function $V(a,w)$ serves both a previously unemployed worker holding offer $w$ and a previously
employed worker holding job $w$. Illustration: $u(c)=1-e^{-c}$, $v(0)=0$, $v(1)=.3$, $z=.3$, $w$ uniform on
$[0,2]$ (15 points), $R\in\{.98,1.06\}$ equally likely, $\beta=.95$, $a\in[0,15]$ (120 points).
Checks: VFI; the value of the VFI policy computed by exact policy evaluation (linear system) equals $V$;
the acceptance set is an upper interval in $w$ for every $a$.

In [14]:
b613, z613, psi613 = .95, .3, .3
R613 = np.array([.98, 1.06]); pR = np.array([.5, .5])
wA, pA = unif_grid(15, 0, 2)
ag = np.linspace(0, 15, 120)
u613 = lambda c: 1 - np.exp(-c)
def interp_w(V, x):
    """Linear interpolation of V(., w) (columns w) at asset levels x (clipped to the grid); returns (len(x), nw)."""
    x = np.clip(x, ag[0], ag[-1])
    j = np.clip(np.searchsorted(ag, x) - 1, 0, len(ag) - 2)
    wgt = (x - ag[j]) / (ag[j + 1] - ag[j])
    return (1 - wgt)[:, None] * V[j] + wgt[:, None] * V[j + 1], j, wgt
sg = ag.copy()                                   # savings grid s = a + income - c
def bellman613(V):
    EVw = sum(pR[r] * interp_w(V, R613[r] * sg)[0] for r in range(2))    # (ns, nw): keep job w
    EVu = EVw @ pA                                                         # (ns,): new offer next period
    cash_w = ag[:, None] + wA[None, :]                                     # (na, nw)
    cw = cash_w[:, :, None] - sg[None, None, :]
    valw = np.where(cw >= 0, u613(np.maximum(cw, 0)) + b613 * EVw.T[None, :, :], -np.inf)
    cu = ag[:, None] + z613 - sg[None, :]
    valu = np.where(cu >= 0, u613(np.maximum(cu, 0)) + psi613 + b613 * EVu[None, :], -np.inf)
    Vw, sw = valw.max(2), valw.argmax(2)
    Vu, su = valu.max(1), valu.argmax(1)
    return np.maximum(Vw, Vu[:, None]), Vw >= Vu[:, None], sw, su
V = np.zeros((len(ag), len(wA)))
for it in range(5000):
    Vn, work, sw, su = bellman613(V)
    if np.max(np.abs(Vn - V)) < 1e-10: break
    V = Vn
# exact evaluation of the VFI policy: V = r + beta P V
na, nw = V.shape; N = na * nw
idx = lambda i, k: i * nw + k
rvec = np.zeros(N); P = np.zeros((N, N))
for i in range(na):
    for k in range(nw):
        s_i = sw[i, k] if work[i, k] else su[i]
        cons = ag[i] + (wA[k] if work[i, k] else z613) - sg[s_i]
        rvec[idx(i, k)] = u613(cons) + (0 if work[i, k] else psi613)
        for r in range(2):
            x = min(R613[r] * sg[s_i], ag[-1]); j = min(max(np.searchsorted(ag, x) - 1, 0), na - 2)
            wt_ = (x - ag[j]) / (ag[j + 1] - ag[j])
            targets = [k] if work[i, k] else range(nw)
            for kk in targets:
                pk = 1. if work[i, k] else pA[kk]
                P[idx(i, k), idx(j, kk)] += pR[r] * pk * (1 - wt_)
                P[idx(i, k), idx(j + 1, kk)] += pR[r] * pk * wt_
Vpol = np.linalg.solve(np.eye(N) - b613 * P, rvec).reshape(na, nw)
assert np.max(np.abs(Vpol - Vn)) < 1e-6
assert np.all(np.diff(work.astype(int), axis=1) >= 0)     # reservation-wage form for every a
wbar613 = np.array([wA[work[i]][0] if work[i].any() else np.inf for i in range(na)])
print(f"VFI iterations: {it}; reservation wage at a = 0, 5, 10, 15:", wbar613[[0, 40, 80, 119]])
fin = np.isfinite(wbar613)
assert np.all(np.diff(fin.astype(int)) <= 0) and np.all(np.diff(wbar613[fin]) >= 0)   # rises with a, then 'never work'
print(f"never works (quits any job) once a >= {ag[~fin][0]:.3f}")

VFI iterations: 446; reservation wage at a = 0, 5, 10, 15: [1.266667 1.933333      inf      inf]
never works (quits any job) once a >= 7.563


## Exercise 6.14 --- temporary unemployment compensation

(a)--(c): $\bar w_+$ solves (6.3.3) with $c=0$ and $\bar w_1=\bar w_++(1-\beta)\gamma$. (d)--(e), with UI after quits:
$\bar w^u_+=0$, $\bar w^u_1=k^{-1}(\gamma)$ with $k(x)=x+\beta\int_0^xF$, $\bar w^e=(1-\beta)\int v_1^u dF$.
$F$ uniform on $[0,1]$, $\beta=.95$, $\gamma\in\{.05,.1,.2\}$. Checks: VFI on all value functions; simulated hazards.

In [15]:
b614 = .95
res614 = {}
for gam in (.05, .1, .2):
    wplus = resw(0., b614, wu, pu)
    w1st = wplus + (1 - b614) * gam
    v1, vp = wu / (1 - b614), wu / (1 - b614)
    for _ in range(100000):
        Ep = pu @ vp
        v1n = np.maximum(wu / (1 - b614), gam + b614 * Ep); vpn = np.maximum(wu / (1 - b614), b614 * Ep)
        if max(np.max(np.abs(v1n - v1)), np.max(np.abs(vpn - vp))) < 1e-12: break
        v1, vp = v1n, vpn
    assert abs((1 - b614) * (gam + b614 * pu @ vpn) - w1st) < 1e-9 and abs((1 - b614) * b614 * pu @ vpn - wplus) < 1e-9
    haz = np.array([pu[wu >= w1st].sum()] + [pu[wu >= wplus].sum()] * 3)
    # simulated hazard by duration
    N = 200000; alive = np.ones(N, bool); hz = []
    for d in range(4):
        draw = wu[rng.integers(0, len(wu), N)]
        acc = alive & (draw >= (w1st if d == 0 else wplus))
        hz.append(acc.sum() / alive.sum()); alive &= ~acc
    assert np.allclose(hz, haz, atol=4e-3)
    # (d) with UI after quits
    ve, u1, up = wu / (1 - b614), wu / (1 - b614), wu / (1 - b614)
    for _ in range(100000):
        E1, Ep = pu @ u1, pu @ up
        A = wu + b614 * ve
        ven, u1n, upn = np.maximum(A, E1), np.maximum(A, gam + b614 * Ep), np.maximum(A, b614 * Ep)
        if max(np.max(np.abs(ven - ve)), np.max(np.abs(u1n - u1)), np.max(np.abs(upn - up))) < 1e-12: break
        ve, u1, up = ven, u1n, upn
    E1, Ep = pu @ u1n, pu @ upn; A = wu + b614 * ven
    assert np.all(A >= b614 * Ep)                                          # wbar_+^u = 0: every offer accepted
    k_inv = optimize.brentq(lambda x: x + b614 * np.sum(pu * np.maximum(x - wu, 0)) - gam, 0, 1, xtol=1e-14)
    acc1 = A >= gam + b614 * Ep
    assert np.array_equal(acc1, wu >= k_inv)
    assert abs(E1 - Ep - np.sum(pu * np.maximum(k_inv - wu, 0))) < 1e-9
    we = (1 - b614) * E1
    stay = A >= E1
    assert np.array_equal(stay, wu >= we) and k_inv < we and k_inv < gam
    # wbar^e = Ew + beta int_0^{wbar^e} F + (1-beta) int_0^{wbar_1^u} F   (int_0^x F = E(x-w)^+)
    assert abs(we - (pu @ wu + b614 * np.sum(pu * np.maximum(we - wu, 0)) + (1 - b614) * np.sum(pu * np.maximum(k_inv - wu, 0)))) < 1e-9
    res614[gam] = (wplus, w1st, k_inv, we, haz[:2])
    print(f"gamma={gam}: (a-c) wbar_1={w1st:.4f}, wbar_+={wplus:.4f}, hazards {haz[0]:.3f} then {haz[1]:.3f}"
          f" | (d-e) wbar_+^u=0, wbar_1^u={k_inv:.4f}, wbar^e={we:.4f}")

gamma=0.05: (a-c) wbar_1=0.7264, wbar_+=0.7239, hazards 0.275 then 0.275 | (d-e) wbar_+^u=0, wbar_1^u=0.0489, wbar^e=0.8175
gamma=0.1: (a-c) wbar_1=0.7289, wbar_+=0.7239, hazards 0.270 then 0.275 | (d-e) wbar_+^u=0, wbar_1^u=0.0957, wbar^e=0.8183
gamma=0.2: (a-c) wbar_1=0.7339, wbar_+=0.7239, hazards 0.265 then 0.275 | (d-e) wbar_+^u=0, wbar_1^u=0.1839, wbar^e=0.8211


## Exercise 6.15 --- seasons, I

$v_o(w)=\max\{w/(1-\beta),\,c+\beta\int v_e\,dF^2\}$ (odd: one offer now, two next period),
$v_e(w)=\max\{w/(1-\beta),\,c+\beta\int v_o\,dF\}$. $F$ uniform on $[0,1]$, $c=.2$, $\beta=.95$.
Checks: two-equation root in $(Q_o,Q_e)$ vs. VFI on both functions; $\bar w_o>\bar w_e$; both lie between the
always-one-offer and always-two-offers reservation wages.

In [16]:
b615, c615 = .95, .2
pu2 = max_dist(pu, 2)
def res615(Q):
    Qo, Qe = Q
    return [Qo - c615 - b615 * pu2 @ np.maximum(wu / (1 - b615), Qe),
            Qe - c615 - b615 * pu @ np.maximum(wu / (1 - b615), Qo)]
Qo, Qe = optimize.fsolve(res615, [10., 10.], xtol=1e-14)
vo, ve = wu / (1 - b615), wu / (1 - b615)
for _ in range(100000):
    von = np.maximum(wu / (1 - b615), c615 + b615 * pu2 @ ve); ven = np.maximum(wu / (1 - b615), c615 + b615 * pu @ vo)
    if max(np.max(np.abs(von - vo)), np.max(np.abs(ven - ve))) < 1e-12: break
    vo, ve = von, ven
wbo, wbe = (1 - b615) * Qo, (1 - b615) * Qe
assert abs(wbo - (1 - b615) * (c615 + b615 * pu2 @ ven)) < 1e-9 and abs(wbe - (1 - b615) * (c615 + b615 * pu @ von)) < 1e-9
w_one, w_two = resw(c615, b615, wu, pu), resw(c615, b615, wu, pu2)
assert w_one < wbe < wbo < w_two
print(f"odd-period wbar={wbo:.5f} > even-period wbar={wbe:.5f}; stationary one offer {w_one:.5f}, two offers {w_two:.5f}")

odd-period wbar=0.79794 > even-period wbar=0.78744; stationary one offer 0.75770, two offers 0.81434


## Exercise 6.16 --- seasons, II

$J_o(w)=w+\beta J_e(w)$, $J_e(w)=w+\beta[(1-\pi)J_o(w)+\pi\int v_o dF]$, $v_o=\max\{J_o,\,c+\beta\int v_e dF\}$,
$v_e=\max\{J_e,\,c+\beta\int v_o dF\}$. $F$ uniform on $[0,1]$, $\beta=.95$. Checks: VFI on four functions vs.
the closed form of $J$ plus a two-equation root; the identity $\bar w_o-\bar w_e=\frac{\beta}{1+\beta}[G(\bar w_e)-\kappa G(\bar w_o)]$, $\kappa=\frac{(1-\pi)(1+\beta)}{1+\beta(1-\pi)}$, over a grid of $(\pi,c)$.

In [17]:
b616 = .95
def solve616(pi_, c):
    Jo, Je, vo, ve = [wu / (1 - b616)] * 4
    for _ in range(200000):
        Eo, Ee = pu @ vo, pu @ ve
        Jon = wu + b616 * Je; Jen = wu + b616 * ((1 - pi_) * Jo + pi_ * Eo)
        von = np.maximum(Jon, c + b616 * Ee); ven = np.maximum(Jen, c + b616 * Eo)
        if max(np.max(np.abs(x - y)) for x, y in ((Jon, Jo), (Jen, Je), (von, vo), (ven, ve))) < 1e-12: break
        Jo, Je, vo, ve = Jon, Jen, von, ven
    Eo, Ee = pu @ von, pu @ ven
    d = 1 - b616**2 * (1 - pi_)
    Jef = lambda w, Eo: (w * (1 + b616 * (1 - pi_)) + b616 * pi_ * Eo) / d
    def r(E):
        Eo_, Ee_ = E
        return [Eo_ - pu @ np.maximum(wu + b616 * Jef(wu, Eo_), c + b616 * Ee_),
                Ee_ - pu @ np.maximum(Jef(wu, Eo_), c + b616 * Eo_)]
    Eo2, Ee2 = optimize.fsolve(r, [Eo + 1, Ee + 1], xtol=1e-14)
    assert abs(Eo2 - Eo) < 1e-7 and abs(Ee2 - Ee) < 1e-7
    # reservation wages from the (linear, increasing) J's
    wbo = optimize.brentq(lambda w: w + b616 * Jef(w, Eo) - c - b616 * Ee, -50, 50)
    wbe = optimize.brentq(lambda w: Jef(w, Eo) - c - b616 * Eo, -50, 50)
    assert np.array_equal(Jon >= c + b616 * Ee, wu >= wbo) and np.array_equal(Jen >= c + b616 * Eo, wu >= wbe)
    return wbo, wbe
wbo616, wbe616 = solve616(.3, .2)
print(f"pi=.3, c=.2: odd-period wbar={wbo616:.5f}, even-period wbar={wbe616:.5f}")
for pi_ in (.1, .5, .9):
    for c in (0., .4, .8):
        o, e = solve616(pi_, c)
        kap = (1 - pi_) * (1 + b616) / (1 + b616 * (1 - pi_))
        # identity  wbar_o - wbar_e = beta/(1+beta) [G(wbar_e) - kappa G(wbar_o)]  and its consequence wbar_o > wbar_e
        assert abs((o - e) - b616 / (1 + b616) * (G(e, wu, pu) - kap * G(o, wu, pu))) < 1e-8 and o > e
        print(f"  pi={pi_}, c={c}: wbar_o={o:.5f}, wbar_e={e:.5f}, difference {o - e:+.5f}")

pi=.3, c=.2: odd-period wbar=0.57077, even-period wbar=0.56051
  pi=0.1, c=0.0: wbar_o=0.62989, wbar_e=0.62770, difference +0.00219
  pi=0.1, c=0.4: wbar_o=0.73131, wbar_e=0.73022, difference +0.00109
  pi=0.1, c=0.8: wbar_o=0.87338, wbar_e=0.87316, difference +0.00022
  pi=0.5, c=0.0: wbar_o=0.40745, wbar_e=0.36611, difference +0.04134
  pi=0.5, c=0.4: wbar_o=0.58935, wbar_e=0.57185, difference +0.01750


  pi=0.5, c=0.8: wbar_o=0.83101, wbar_e=0.82844, difference +0.00257
  pi=0.9, c=0.0: wbar_o=0.26893, wbar_e=0.09071, difference +0.17822
  pi=0.9, c=0.4: wbar_o=0.50462, wbar_e=0.43846, difference +0.06617
  pi=0.9, c=0.8: wbar_o=0.81287, wbar_e=0.80514, difference +0.00773


## Exercise 6.17 --- Gittins indexes for beginners

(a) $V(i,I)=\max\{\sum_jA_{ij}[w_a(j)+\beta V(j,I)],\ \sum_JB_{IJ}[w_b(J)+\beta V(i,J)]\}$.
(b) $v_a(i)=\max\{\sum_jA_{ij}[w_a(j)+\beta v_a(j)],\ \sum_J\pi_{bJ}[w_b(J)+\beta V(i,J)]\}$, $v_b$ symmetric; start:
$\max\{\sum_i\pi_{ai}[w_a(i)+\beta v_a(i)],\ \sum_I\pi_{bI}[w_b(I)+\beta v_b(I)]\}$.
Example: $n=4$ states per job, random (seeded) $A$, $B$, wages, $\pi_a,\pi_b$; $\beta=.9$.
Independent route: Gittins indexes by the restart-in-state formulation (Katehakis--Veinott), one arm at a
time; the Gittins theorem says the optimal choice is the job with the larger index. Part (b) is also solved by
adding a "fresh" state 0 (row $\pi_a$) to each chain.

In [18]:
r617 = np.random.default_rng(1)
n617, b617 = 4, .9
A617 = r617.dirichlet(np.ones(n617), n617); B617 = r617.dirichlet(np.ones(n617), n617)
wa617 = np.sort(r617.uniform(0, 10, n617)); wb617 = np.sort(r617.uniform(0, 10, n617))
pia, pib = r617.dirichlet(np.ones(n617)), r617.dirichlet(np.ones(n617))
def solve_two_jobs(A, wa, B, wb, beta):
    V = np.zeros((len(wa), len(wb)))
    for _ in range(100000):
        goA = (A @ (wa[:, None] + beta * V))          # (i, I): sum_j A_ij [wa_j + beta V(j, I)]
        goB = ((wb[None, :] + beta * V) @ B.T)        # (i, I): sum_J B_IJ [wb_J + beta V(i, J)]
        Vn = np.maximum(goA, goB)
        if np.max(np.abs(Vn - V)) < 1e-12: return Vn, goA, goB
        V = Vn
def gittins(P, w, beta):
    """Index of every state of one arm: restart-in-k problem, index = (1-beta) M_k(k)."""
    r = P @ w; idx = np.zeros(len(w))
    for k in range(len(w)):
        M = np.zeros(len(w))
        for _ in range(100000):
            Mn = np.maximum(r + beta * P @ M, r[k] + beta * P[k] @ M)
            if np.max(np.abs(Mn - M)) < 1e-13: break
            M = Mn
        idx[k] = (1 - beta) * Mn[k]
    return idx
V617, goA, goB = solve_two_jobs(A617, wa617, B617, wb617, b617)
ga, gb = gittins(A617, wa617, b617), gittins(B617, wb617, b617)
for i in range(n617):
    for I in range(n617):
        if abs(ga[i] - gb[I]) > 1e-8:
            assert (goA[i, I] > goB[i, I]) == (ga[i] > gb[I])
print("Gittins indexes job A:", ga, "\n                 job B:", gb)
print("choose A at (i, I):\n", (goA > goB).astype(int))
# (b) hint equations, iterated with V from (a)
va, vb = np.zeros(n617), np.zeros(n617)
for _ in range(100000):
    van = np.maximum(A617 @ (wa617 + b617 * va), pib @ (wb617[:, None] + b617 * V617.T))
    vbn = np.maximum(B617 @ (wb617 + b617 * vb), pia @ (wa617[:, None] + b617 * V617))
    if max(np.max(np.abs(van - va)), np.max(np.abs(vbn - vb))) < 1e-12: break
    va, vb = van, vbn
startA, startB = pia @ (wa617 + b617 * van), pib @ (wb617 + b617 * vbn)
# augmented chains with a fresh state 0
aug = lambda P, pi0: np.vstack([np.concatenate([[0.], pi0]), np.hstack([np.zeros((len(pi0), 1)), P])])
Aa, Ba = aug(A617, pia), aug(B617, pib)
wa_a, wb_a = np.concatenate([[0.], wa617]), np.concatenate([[0.], wb617])
Vaug, gA_aug, gB_aug = solve_two_jobs(Aa, wa_a, Ba, wb_a, b617)
assert np.allclose(Vaug[1:, 0], van) and np.allclose(Vaug[0, 1:], vbn) and abs(Vaug[0, 0] - max(startA, startB)) < 1e-9
gfa, gfb = gittins(Aa, wa_a, b617)[0], gittins(Ba, wb_a, b617)[0]
assert (startA > startB) == (gfa > gfb)
print(f"new worker: start at {'A' if startA > startB else 'B'} (values {startA:.4f} vs {startB:.4f}; fresh indexes {gfa:.4f} vs {gfb:.4f})")

Gittins indexes job A: [5.858443 5.985309 6.075019 5.278259] 
                 job B: [6.288919 5.925875 4.898593 4.594895]
choose A at (i, I):
 [[0 0 1 1]
 [0 1 1 1]
 [0 1 1 1]
 [0 0 1 1]]
new worker: start at B (values 55.7047 vs 55.7796; fresh indexes 5.8515 vs 6.2636)


## Exercise 6.18 --- Jovanovic (1979b)

The printed law of motion $x_{t+1}=g(x_t\phi_t)-\delta x_t$ makes capital negative whenever $\phi_t$ is small;
we use the depreciation form $x_{t+1}=(1-\delta)x_t+g(x_t\phi_t)$. Bellman equation:
$V(x)=\max_{\phi,s}\{x(1-\phi-s)+\beta[\pi(s)\sum_if_i\max\{V(\mu_i),V(x')\}+(1-\pi(s))V(x')]\}$.
With $\pi(s)=\sqrt s$ the search choice has a closed form given $\phi$: $s=\min\{(\beta K(x')/2x)^2,\,1-\phi\}$, with
$K(x')=\sum_if_i\max\{V(\mu_i)-V(x'),0\}$. Parameters: $g(x\phi)=(x\phi)^{.5}$ ($A=1$, $\alpha=.5$), $\delta=.1$,
$\beta=.95$, $\mu\in\{5,10,\dots,45\}$ with $f_i=1/9$; $s\le .999-\phi$ keeps $\pi(s)<1$. Checks: $V$ nondecreasing (so accept $\mu'$ iff $\mu'>x'$);
brute-force $(\phi,s)$ grid at sample states; Monte Carlo value of the computed policy vs. $V$.

In [19]:
b618, al618, dl618, A618 = .95, .5, .1, 1.
mu618 = np.arange(5., 46., 5.); f618 = np.full(mu618.size, 1 / mu618.size)
xg = np.linspace(.5, 100, 300)
phg = np.linspace(0, .998, 400)
gfun = lambda y: A618 * y**al618
def step(V, x, full=False):
    """One Bellman step at states x given V on xg; returns value (and argmax phi, s)."""
    xp = (1 - dl618) * x[:, None] + gfun(x[:, None] * phg[None, :])        # (nx, nphi)
    Vxp = np.interp(xp, xg, V)
    K = np.tensordot(f618, np.maximum(np.interp(mu618, xg, V)[:, None, None] - Vxp[None], 0), axes=(0, 0))
    s = np.minimum((b618 * K / (2 * x[:, None]))**2, .999 - phg[None, :])   # keeps pi(s) < 1
    val = x[:, None] * (1 - phg[None, :] - s) + b618 * (Vxp + np.sqrt(s) * K)
    j = val.argmax(1)
    if full:
        return val[np.arange(len(x)), j], phg[j], s[np.arange(len(x)), j]
    return val[np.arange(len(x)), j]
V618 = np.zeros(xg.size)
for it in range(5000):
    Vn = step(V618, xg)
    if np.max(np.abs(Vn - V618)) < 1e-9: break
    V618 = Vn
V618, phi618, s618 = step(Vn, xg, full=True)
assert np.all(np.diff(V618) > 0)
full = xg[s618 >= .999 - phi618 - 1e-12]
print(f"VFI iterations {it}; V strictly increasing; search full time (phi=0, s=.999) for x <= {full.max():.2f}")
# brute force (phi, s) grid at a few states
for x in (3., 20., 60.):
    P_, S_ = np.meshgrid(np.linspace(0, .998, 300), np.linspace(0, .999, 300), indexing="ij")
    ok = P_ + S_ <= .999
    xp = (1 - dl618) * x + gfun(x * P_)
    Vxp = np.interp(xp, xg, V618)
    Emax = np.tensordot(f618, np.maximum(np.interp(mu618, xg, V618)[:, None, None], Vxp[None]), axes=(0, 0))
    val = np.where(ok, x * (1 - P_ - S_) + b618 * (np.sqrt(S_) * Emax + (1 - np.sqrt(S_)) * Vxp), -np.inf)
    assert abs(val.max() - np.interp(x, xg, V618)) < 2e-3 * np.interp(x, xg, V618)
# Monte Carlo value of the policy, starting from x0
def simulate(x0, N=2000, T=200):
    x = np.full(N, x0); tot = np.zeros(N)
    for t in range(T):
        _, ph, s = step(V618, x, full=True)
        tot += b618**t * x * (1 - ph - s)
        xn = (1 - dl618) * x + gfun(x * ph)
        offer = rng.random(N) < np.sqrt(s)
        mu_ = mu618[rng.integers(0, mu618.size, N)]
        x = np.where(offer & (np.interp(mu_, xg, V618) > np.interp(xn, xg, V618)), mu_, xn)
    return tot.mean(), tot.std() / np.sqrt(N)
for x0 in (5., 30.):
    m, se = simulate(x0)
    assert abs(m - np.interp(x0, xg, V618)) < 4 * se + 2e-3 * m
    print(f"x0={x0}: V={np.interp(x0, xg, V618):.4f}, Monte Carlo {m:.4f} (s.e. {se:.4f})")
print("phi(x), s(x) at x=5,20,45,80:", np.interp([5, 20, 45, 80], xg, phi618), np.interp([5, 20, 45, 80], xg, s618))
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.6))
ax[0].plot(xg, phi618, label=r"$\phi(x)$ (investment)"); ax[0].plot(xg, s618, "--", label=r"$s(x)$ (search)")
ax[0].set_xlabel(r"job-specific capital $x$"); ax[0].legend(); ax[0].set_title("Policies")
ax[1].plot(xg, V618); ax[1].set_xlabel(r"$x$"); ax[1].set_title(r"$V(x)$")
for m in mu618: ax[1].axvline(m, color="0.85", lw=.6, zorder=0)
fig.tight_layout(); save_pdf(fig, "jovanovic"); plt.close(fig)

VFI iterations 468; V strictly increasing; search full time (phi=0, s=.999) for x <= 12.15


x0=5.0: V=489.5042, Monte Carlo 491.2198 (s.e. 1.1847)


x0=30.0: V=513.8242, Monte Carlo 514.4485 (s.e. 1.0802)
phi(x), s(x) at x=5,20,45,80: [0.       0.002501 0.129374 0.120312] [0.999    0.323843 0.000164 0.      ]


## Exercise 6.19 --- value function iteration vs. policy improvement

$F$ uniform on $[0,1]$, $c=.3$, $\beta=.95$. VFI: $w_{n+1}=(1-\beta)c+\beta[w_n+\int_{w_n}(w-w_n)dF]$, local rate $\beta F(w^*)$.
Howard: $w_{n+1}=\dfrac{(1-\beta)c+\beta\int_{w_n}w\,dF}{1-\beta F(w_n)}$, $w_{n+1}-w^*\approx K(w_n-w^*)^2$, $K=-\dfrac{\beta f(w^*)}{2(1-\beta F(w^*))}$.
Checks: the scalar recursions vs. VFI / policy evaluation on the whole function (discrete grid); the rates.

In [20]:
b619, c619 = .95, .3
wstar = optimize.brentq(lambda w: w - c619 - b619 / (1 - b619) * (1 - w)**2 / 2, 0, 1, xtol=1e-15)
vfi_seq = [(1 - b619) * c619]
for _ in range(400):
    w = vfi_seq[-1]; vfi_seq.append((1 - b619) * c619 + b619 * (w + (1 - w)**2 / 2))
how_seq = [0.]
for _ in range(8):
    w = how_seq[-1]; how_seq.append(((1 - b619) * c619 + b619 * (1 - w**2) / 2) / (1 - b619 * w))
vfi_seq, how_seq = np.array(vfi_seq), np.array(how_seq)
ev, eh = vfi_seq - wstar, how_seq - wstar
ratio_v = ev[1:] / ev[:-1]
selv = (np.abs(ev[:-1]) > 1e-9) & (np.abs(ev[:-1]) < 1e-5)
assert np.all(np.abs(ratio_v[selv] - b619 * wstar) < 1e-4) and np.all(ratio_v[np.abs(ev[:-1]) > 1e-12] < b619)
Kth = -b619 / (2 * (1 - b619 * wstar))
sel = (np.abs(eh[:-1]) > 1e-5) & (np.abs(eh[:-1]) < 5e-2)
kq = eh[1:][sel] / eh[:-1][sel]**2
assert np.all(np.abs(kq - Kth) < .05 * abs(Kth)) and np.all(eh[1:] <= 1e-15)
# whole-function versions on a discrete grid (n = 4000 midpoints) reproduce the recursions for the same F
wd, pd = unif_grid(4000, 0, 1)
v = np.zeros(wd.size); seq_grid = []
for n in range(6):
    Q = c619 + b619 * pd @ v; seq_grid.append((1 - b619) * Q); v = np.maximum(wd / (1 - b619), Q)
seq_rec = [(1 - b619) * c619]
for _ in range(5):
    w = seq_rec[-1]; seq_rec.append((1 - b619) * c619 + b619 * (w + G(w, wd, pd)))
assert np.allclose(seq_grid, seq_rec, atol=1e-12)
x = 0.; hg = []
for n in range(5):   # Howard on the grid: evaluate 'accept iff w >= x', then improve
    acc = wd >= x
    vpol = np.zeros(wd.size)
    for _ in range(5000):   # policy evaluation by successive approximation on the whole function
        vn = np.where(acc, wd + b619 * vpol, c619 + b619 * pd @ vpol)
        if np.max(np.abs(vn - vpol)) < 1e-13: break
        vpol = vn
    x = (1 - b619) * (c619 + b619 * pd @ vn); hg.append(x)
hrec = [0.]
for _ in range(5):
    w = hrec[-1]; hrec.append(((1 - b619) * c619 + b619 * pd[wd >= w] @ wd[wd >= w]) / (1 - b619 * pd[wd < w].sum()))
assert np.allclose(hg, hrec[1:], atol=1e-12)
print(f"w* = {wstar:.10f}; VFI local rate {ratio_v[selv][-1]:.6f} = beta F(w*) = {b619*wstar:.6f}; Howard K = {Kth:.4f}, "
      f"measured {kq}")
print("Howard iterates:", how_seq[:6], "errors:", eh)
fig, ax = plt.subplots(figsize=(4.2, 2.8))
nv = np.arange(1, 61)
ax.semilogy(nv, np.abs(ev[:60]) + 1e-17, "o-", ms=2.5, label="value iteration")
ax.semilogy(np.arange(len(eh)), np.abs(eh) + 1e-17, "s-", ms=3, label="Howard policy improvement")
ax.set_ylim(1e-16, 1); ax.set_xlabel("iteration $n$"); ax.set_ylabel(r"$|w_n-w^*|$"); ax.legend()
fig.tight_layout(); save_pdf(fig, "howard"); plt.close(fig)

w* = 0.7761278834; VFI local rate 0.737321 = beta F(w*) = 0.737321; Howard K = -1.8083, measured [-1.760057 -1.807633]
Howard iterates: [0.       0.49     0.703372 0.76855  0.776027 0.776128] errors: [-0.776128 -0.286128 -0.072756 -0.007578 -0.000101 -0.       -0.
 -0.       -0.      ]


## Exercise 6.20 --- mean-preserving spreads with atoms

$E_\alpha w=B/2$ for every $\alpha$ and $\int_0^y(F_\alpha-F_0)\ge0$: $F_{.3}$ is an MPS of $F_0$. Reservation wage from (6.3.5):
$\bar w-c=\beta(B/2-c)+\beta g_\alpha(\bar w)$, $g_\alpha(x)=\int_0^xF_\alpha$. $B=1$, $\beta=.9$.
Checks: closed-form $g_\alpha$ vs. VFI on a discretised $F_\alpha$ (atoms at $0$ and $B$); hazards $1-F_\alpha(\bar w^-)$;
the type with the faster exit switches at $c^*$ where $\bar w_0=.7$.

In [21]:
b620, B620 = .9, 1.
def g_alpha(x, a):
    lo, hi = a * B620, B620 - a * B620
    if x <= lo: return a * x
    if x < hi: return a * lo + (x**2 - lo**2) / (2 * B620)
    return a * lo + (hi**2 - lo**2) / (2 * B620) + (1 - a) * (x - hi)
def wbar620(c, a):
    return optimize.brentq(lambda x: x - c - b620 * (B620 / 2 - c) - b620 * g_alpha(x, a), 0, B620, xtol=1e-14)
def disc620(a, n=20000):
    wm, pm = unif_grid(n, a * B620, B620 - a * B620); pm = pm * (1 - 2 * a)
    return np.concatenate([[0.], wm, [B620]]), np.concatenate([[a], pm, [a]])
for a in (0., .3):
    wdisc, pdisc = disc620(a)
    assert abs(pdisc @ wdisc - B620 / 2) < 1e-12
    for c in (.1, .3, .6):
        assert abs(resw(c, b620, wdisc, pdisc) - wbar620(c, a)) < 1e-6
ys = np.linspace(0, 1, 101)
assert all(g_alpha(y, .3) - g_alpha(y, 0.) >= -1e-15 for y in ys)
def hazard(c, a):
    wb = wbar620(c, a)
    Fminus = a if wb <= a * B620 else (wb / B620 if wb <= B620 - a * B620 else 1 - a)
    return 1 - Fminus
cstar = .7 - b620 / (1 - b620) * .3**2 / 2
assert abs(wbar620(cstar, 0.) - .7) < 1e-10
cg = np.linspace(.01, .9, 90)
W0 = np.array([wbar620(c, 0.) for c in cg]); W3 = np.array([wbar620(c, .3) for c in cg])
H0 = np.array([hazard(c, 0.) for c in cg]); H3 = np.array([hazard(c, .3) for c in cg])
assert np.all(W3 > W0)
assert np.all((H0 > H3)[cg < cstar - 1e-9]) and np.all((H3 > H0)[cg > cstar + 1e-9])
# simulated mean durations at two values of c
for c in (.1, .5):
    for a in (0., .3):
        wdisc, pdisc = disc620(a, 2000)
        wb = wbar620(c, a)
        acc = rng.choice(wdisc, size=(20000, 200), p=pdisc) >= wb
        dur = acc.argmax(1) + 1
        assert abs(dur.mean() - 1 / hazard(c, a)) < 4 * dur.std() / np.sqrt(dur.size)
print(f"c* = {cstar:.4f}; at c=.1: wbar = {wbar620(.1,0):.4f} vs {wbar620(.1,.3):.4f}, hazard {hazard(.1,0):.4f} vs {hazard(.1,.3):.4f}")
print(f"          at c=.5: wbar = {wbar620(.5,0):.4f} vs {wbar620(.5,.3):.4f}, hazard {hazard(.5,0):.4f} vs {hazard(.5,.3):.4f}")
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.6))
ax[0].plot(cg, W0, label=r"$\alpha=0$"); ax[0].plot(cg, W3, "--", label=r"$\alpha=.3$")
ax[0].set_xlabel("$c$"); ax[0].set_title(r"reservation wage $\bar w_\alpha$"); ax[0].legend()
ax[1].plot(cg, H0, label=r"$\alpha=0$"); ax[1].plot(cg, H3, "--", label=r"$\alpha=.3$"); ax[1].axvline(cstar, color=".6", lw=.7)
ax[1].set_xlabel("$c$"); ax[1].set_title(r"hazard $1-F_\alpha(\bar w^-)$"); ax[1].legend()
fig.tight_layout(); save_pdf(fig, "mps_atoms"); plt.close(fig)

c* = 0.2950; at c=.1: wbar = 0.6503 vs 0.7568, hazard 0.3497 vs 0.3000
          at c=.5: wbar = 0.7597 vs 0.8649, hazard 0.2403 vs 0.3000


## Exercise 6.21 --- searching for the lowest price

Reservation price $\bar p$: $c=\int_{\underline B}^{\bar p}F(p)\,dp$ (if $c\ge\bar B-Ep$, buy at the first draw: $\bar p=c+Ep$).
Expected price plus search costs $=\bar p$. $F$ uniform on $[1,2]$: $\bar p=1+\sqrt{2c}$ for $c\le .5$.
Checks: minimise $E[p\mid p\le x]+c/F(x)$ over $x$; Monte Carlo of total outlay.

In [22]:
for c in (.02, .1, .6):
    pbar = 1 + np.sqrt(2 * c) if c <= .5 else c + 1.5
    xs = np.linspace(1.0001, 2, 20000)
    cost = (1 + xs) / 2 + c / (xs - 1)
    if c <= .5:
        assert abs(xs[cost.argmin()] - pbar) < 1e-3 and abs(cost.min() - pbar) < 1e-6
    else:
        assert xs[cost.argmin()] == xs[-1] and abs(cost.min() - pbar) < 1e-6
    N = 200000; p = 1 + rng.random(N); draws = np.ones(N); left = p > pbar
    while left.any():
        p[left] = 1 + rng.random(left.sum()); draws[left] += 1; left = p > pbar
    tot = p + c * draws
    assert abs(tot.mean() - pbar) < 4 * tot.std() / np.sqrt(N)
    print(f"c={c}: pbar={pbar:.5f}, simulated outlay {tot.mean():.5f}, mean draws {draws.mean():.3f}")

c=0.02: pbar=1.20000, simulated outlay 1.19991, mean draws 4.999
c=0.1: pbar=1.44721, simulated outlay 1.44823, mean draws 2.240
c=0.6: pbar=2.10000, simulated outlay 2.10069, mean draws 1.000


## Exercise 6.22 --- quits after reclassification

$W(w)=w+\beta[(1-\alpha)W(w)+\alpha\int v\,dF]$, $v(w)=\max\{W(w),\,c+\beta\int v\,dF\}$; the same $\bar w$ governs acceptance
and quits: $\bar w-c=\frac{\beta(1-\alpha)}{1-\beta(1-\alpha)}\int_{\bar w}(w-\bar w)dF$; quit probability $\alpha F(\bar w)$.
$F$ uniform on $[0,1]$, $c=.3$, $\beta=.95$, $\alpha=.2$. Checks: VFI on $(W,v)$ arrays; simulated quit frequency.

In [23]:
b622, c622, al622 = .95, .3, .2
scale = (1 - al622) * (1 - b622) / (1 - b622 * (1 - al622))
wb622 = resw(c622, b622, wu, pu, scale=scale)
W, v = wu / (1 - b622), wu / (1 - b622)
for _ in range(100000):
    Ev = pu @ v
    Wn = wu + b622 * ((1 - al622) * W + al622 * Ev); vn = np.maximum(Wn, c622 + b622 * Ev)
    if max(np.max(np.abs(Wn - W)), np.max(np.abs(vn - v))) < 1e-12: break
    W, v = Wn, vn
accept = Wn >= c622 + b622 * pu @ vn
assert np.array_equal(accept, wu >= wb622)
# simulate employed workers (decisions read off the VFI arrays, not off wbar)
N, Tq = 50000, 200
k = rng.choice(np.flatnonzero(accept), N); quits = spells = 0
for t in range(Tq):
    recl = rng.random(N) < al622
    knew = rng.integers(0, wu.size, N)
    q = recl & ~accept[knew]
    quits += q.sum(); spells += N
    k = np.where(recl & accept[knew], knew, k)
    k[q] = rng.choice(np.flatnonzero(accept), q.sum())      # replace quitters by new hires
qf = quits / spells
assert abs(qf - al622 * pu[wu < wb622].sum()) < 4 * np.sqrt(qf * (1 - qf) / spells)
print(f"wbar={wb622:.5f}; quit probability alpha F(wbar)={al622 * pu[wu < wb622].sum():.5f}, simulated {qf:.5f}")

wbar=0.57970; quit probability alpha F(wbar)=0.11600, simulated 0.11607


## Exercise 6.23 --- a career ladder

$W(w)=w+\beta[(1-\alpha)W(w)+\alpha W(\gamma w)]\Rightarrow W(w)=w/\kappa$, $\kappa=1-\beta(1-\alpha+\alpha\gamma)>0$.
$v(w)=\max\{w/\kappa,\ c+\beta\int v\,dF\}$; $(1-\beta)\bar w=\kappa c+\beta\int_{\bar w}(w-\bar w)dF$.
$F$ uniform on $[0,1]$, $c=.3$, $\beta=.95$, $\alpha=.1$, $\gamma=1.2$. Checks: $W$ by Monte Carlo; $\bar w$ by VFI.

In [24]:
b623, c623, al623, g623 = .95, .3, .1, 1.2
kap623 = 1 - b623 * (1 - al623 + al623 * g623)
N, T = 20000, 900
wage = np.ones(N); tot = np.zeros(N)
for t in range(T):
    tot += b623**t * wage
    wage = np.where(rng.random(N) < al623, g623 * wage, wage)
assert abs(tot.mean() - 1 / kap623) < 4 * tot.std() / np.sqrt(N) + b623**T * 50
wb623 = optimize.brentq(lambda x: (1 - b623) * x - kap623 * c623 - b623 * G(x, wu, pu), -1, 2, xtol=1e-14)
v = wu / kap623
for _ in range(100000):
    vn = np.maximum(wu / kap623, c623 + b623 * pu @ v)
    if np.max(np.abs(vn - v)) < 1e-12: break
    v = vn
assert abs(kap623 * (c623 + b623 * pu @ vn) - wb623) < 1e-9
wb_noprom = resw(c623, b623, wu, pu)
assert wb623 < wb_noprom
print(f"W(1) = 1/kappa = {1/kap623:.4f} (Monte Carlo {tot.mean():.4f}); wbar = {wb623:.5f} < McCall {wb_noprom:.5f}")

W(1) = 1/kappa = 32.2581 (Monte Carlo 32.3025); wbar = 0.75522 < McCall 0.77613


## Exercise 6.24 --- human capital

$S(w,h)=wh+\beta\sum_jH^e(h,j)V^e(w,h_j)$ (stay), $V^e(w,h)=\max\{S(w,h),U(h)\}$, $U(h)=\int\max\{S(w,h),\,c+\beta\sum_jH^u(h,j)U(h_j)\}dF(w)$.
Example: $h\in\{1,1.25,\dots,2\}$; $H^e$: up .4, stay .5, down .1; $H^u$: down .4, stay .5, up .1 (reflecting at the ends);
$w$ uniform on $[0,1]$ (100 points); $c=.5$, $\beta=.95$. Checks: VFI vs. Howard policy iteration (exact linear
policy evaluation); quit thresholds exceed acceptance thresholds; simulated quit rate.

In [25]:
hs = np.linspace(1, 2, 5); nh = hs.size
def tri(up, stay, down):
    H = np.zeros((nh, nh))
    for i in range(nh):
        H[i, i] += stay; H[i, min(i + 1, nh - 1)] += up; H[i, max(i - 1, 0)] += down
    return H
He, Hu = tri(.4, .5, .1), tri(.1, .5, .4)
wh_, ph_ = unif_grid(100, 0, 1); c624, b624 = .5, .95
earn = wh_[:, None] * hs[None, :]
Ve, U = np.zeros((wh_.size, nh)), np.zeros(nh)
for it in range(100000):
    S = earn + b624 * Ve @ He.T
    rej = c624 + b624 * Hu @ U
    Un = ph_ @ np.maximum(S, rej[None, :]); Ven = np.maximum(S, Un[None, :])
    if max(np.max(np.abs(Ven - Ve)), np.max(np.abs(Un - U))) < 1e-11: break
    Ve, U = Ven, Un
S = earn + b624 * Ven @ He.T; rej = c624 + b624 * Hu @ Un
acc, stay = S >= rej[None, :], S >= Un[None, :]
# Howard: evaluate (acc, stay) exactly, improve, repeat
nw_ = wh_.size
def evaluate(acc, stay):
    n = nw_ * nh + nh; M = np.eye(n); r = np.zeros(n)
    iS = lambda k, j: k * nh + j; iU = lambda j: nw_ * nh + j
    for k in range(nw_):
        for j in range(nh):
            r[iS(k, j)] = earn[k, j]
            for jj in range(nh):
                if stay[k, jj]: M[iS(k, j), iS(k, jj)] -= b624 * He[j, jj]
                else: M[iS(k, j), iU(jj)] -= b624 * He[j, jj]
    for j in range(nh):
        for k in range(nw_):
            if acc[k, j]: M[iU(j), iS(k, j)] -= ph_[k]
            else:
                r[iU(j)] += ph_[k] * c624
                for jj in range(nh): M[iU(j), iU(jj)] -= ph_[k] * b624 * Hu[j, jj]
    x = np.linalg.solve(M, r)
    return x[:nw_ * nh].reshape(nw_, nh), x[nw_ * nh:]
a_, s_ = np.ones_like(acc), np.ones_like(stay)
for hit in range(50):
    Sp, Up = evaluate(a_, s_)
    an, sn = Sp >= (c624 + b624 * Hu @ Up)[None, :], Sp >= Up[None, :]
    if np.array_equal(an, a_) and np.array_equal(sn, s_): break
    a_, s_ = an, sn
assert np.array_equal(a_, acc) and np.array_equal(s_, stay) and np.allclose(Up, Un, atol=1e-8)
assert np.all(np.diff(acc.astype(int), axis=0) >= 0) and np.all(np.diff(stay.astype(int), axis=0) >= 0)
wacc = np.array([wh_[acc[:, j]][0] for j in range(nh)]); wstay = np.array([wh_[stay[:, j]][0] for j in range(nh)])
assert np.all(wstay >= wacc)
# simulated quits among employed workers
N = 20000; k = rng.integers(0, nw_, N); j = np.full(N, 2); emp = np.zeros(N, bool); nq = ne = 0
for t in range(400):
    # employed: draw new h from He, then decide to stay or quit (quitters draw immediately)
    if t:
        cum = np.where(emp[:, None], np.cumsum(He, 1)[j], np.cumsum(Hu, 1)[j])
        j = np.minimum((rng.random(N)[:, None] > cum).sum(1), nh - 1)
    quit = emp & ~stay[k, j]
    if t > 100: nq += quit.sum(); ne += emp.sum()
    search = ~emp | quit
    knew = rng.integers(0, nw_, N)
    k = np.where(search, knew, k)
    emp = np.where(search, acc[knew, j], True)
print(f"VFI {it} iterations, Howard {hit} improvements; acceptance thresholds {wacc}, quit thresholds {wstay}")
print(f"U(h) = {Un}; simulated quit rate among employed: {nq / ne:.4f}")
assert nq > 0

VFI 507 iterations, Howard 6 improvements; acceptance thresholds [0.165 0.005 0.005 0.065 0.185], quit thresholds [0.845 0.835 0.825 0.825 0.815]
U(h) = [27.865525 29.060787 30.271789 31.169342 31.660566]; simulated quit rate among employed: 0.0006


## Exercise 6.25 --- Markov wages

$v_i=\max\{\bar w_i/(1-\beta),\ c+\beta\sum_jP_{ij}v_j\}$. Checks: VFI vs. exhaustive enumeration of all $2^5$
accept/reject rules (each evaluated by a linear solve); the optimal value dominates every rule's value.

In [26]:
P625 = np.array([[.8, .2, 0, 0, 0], [.18, .8, .02, 0, 0], [.25, .25, 0, .25, .25], [0, 0, .02, .8, .18], [0, 0, 0, .2, .8]])
wv625 = np.arange(1., 6.); c625 = 1.
assert np.allclose(P625.sum(1), 1)
res625 = {}
for beta in (.95, .99):
    v = wv625 / (1 - beta)
    for _ in range(200000):
        vn = np.maximum(wv625 / (1 - beta), c625 + beta * P625 @ v)
        if np.max(np.abs(vn - v)) < 1e-10: break
        v = vn
    pol = wv625 / (1 - beta) >= c625 + beta * P625 @ vn - 1e-9
    best, allv = None, []
    for m in range(32):
        a = np.array([(m >> i) & 1 for i in range(5)], bool)
        Mx = np.eye(5) - beta * P625 * (~a)[:, None]
        val = np.linalg.solve(Mx, np.where(a, wv625 / (1 - beta), c625))
        allv.append(val)
        if np.all(val >= vn - 1e-7): best = a
    assert best is not None and np.array_equal(best, pol)
    assert np.all(np.array(allv) <= vn + 1e-7)
    res625[beta] = (pol, vn)
    print(f"beta={beta}: accept = {pol.astype(int)}, v = {vn}")
assert not np.all(np.diff(res625[.99][0].astype(int)) >= 0) or not np.all(np.diff(res625[.95][0].astype(int)) >= 0)

beta=0.95: accept = [0 1 0 0 1], v = [ 35.833333  40.        61.834492  80.311897 100.      ]
beta=0.99: accept = [0 0 0 0 1], v = [224.187691 230.459797 352.797534 466.756688 500.      ]


## Exercise 6.26 --- Neal model's Markov implications

Neal's model as in the text: $\theta,\epsilon$ discrete uniform on 50 evenly spaced points of $[0,5]$, $\beta=.95$.
$P_{12}=\Pr\{\theta'\ge\bar\theta,\epsilon'<\bar\epsilon\}$, $P_{13}=\Pr\{(\theta',\epsilon')\in\text{stay region}\}$, $P_{23}=1-G(\bar\epsilon^-)$.
Checks: $P_{2j}$ is the same for every $\theta\ge\bar\theta$; $P$ vs. transition frequencies from a simulated panel
(which is also the estimator of part d).

In [27]:
th = np.linspace(0, 5, 50); ep = th.copy(); q = np.full(50, 1 / 50); b626 = .95
def neal(c=None, beta=b626):
    """VFI for Neal's model (c=None) or the version with unemployment of Exercise 6.27 (flow c while searching)."""
    v = (th[:, None] + ep[None, :]) / (1 - beta)
    for _ in range(100000):
        stay = th[:, None] + ep[None, :] + beta * v
        if c is None:
            newjob = th[:, None] + ((ep[None, :] + beta * v) @ q)[:, None]    # (nth, 1)
            newlife = q @ (th[:, None] + ep[None, :] + beta * v) @ q
        else:
            newjob = c + beta * (v @ q)[:, None]
            newlife = c + beta * q @ v @ q
        vn = np.maximum(np.maximum(stay, newjob + 0 * stay), newlife)
        if np.max(np.abs(vn - v)) < 1e-10: break
        v = vn
    stay = th[:, None] + ep[None, :] + beta * vn
    if c is None:
        newjob = th[:, None] + ((ep[None, :] + beta * vn) @ q)[:, None] + 0 * stay; newlife = q @ (th[:, None] + ep[None, :] + beta * vn) @ q
    else:
        newjob = c + beta * (vn @ q)[:, None] + 0 * stay; newlife = c + beta * q @ vn @ q
    region = np.where(stay >= np.maximum(newjob, newlife) - 1e-9, 3, np.where(newjob >= newlife - 1e-9, 2, 1))
    return vn, region, newjob[:, 0], newlife
v626, reg626, C626, Q626 = neal()
it_bar = np.argmax(C626 >= Q626 - 1e-9); thbar = th[it_bar]
assert np.all(C626[it_bar:] >= Q626 - 1e-9) and np.all(C626[:it_bar] < Q626)
P12 = q @ (reg626 == 2) @ q; P13 = q @ (reg626 == 3) @ q
P23s = np.array([(reg626[i] == 3) @ q for i in range(it_bar, 50)])
assert np.allclose(P23s, P23s[0]) and np.all(reg626[it_bar:] != 1)
P23 = P23s[0]; epsbar = ep[reg626[-1] == 3][0]
P626 = np.array([[1 - P12 - P13, P12, P13], [0, 1 - P23, P23], [0, 0, 1]])
print(f"theta_bar = {thbar:.4f}, eps_bar = {epsbar:.4f}\nP =\n{P626}")
# simulated panel: N workers, 40 periods, all start in 'new life'
N, Tn = 5000, 40
i_th, i_ep = rng.integers(0, 50, N), rng.integers(0, 50, N)
S_hist = np.zeros((Tn, N), int)
for t in range(Tn):
    s = reg626[i_th, i_ep]; S_hist[t] = s
    life, job = s == 1, s == 2
    i_th = np.where(life, rng.integers(0, 50, N), i_th)
    i_ep = np.where(life | job, rng.integers(0, 50, N), i_ep)
cnt = np.zeros((3, 3))
np.add.at(cnt, (S_hist[:-1].ravel() - 1, S_hist[1:].ravel() - 1), 1)
Phat = cnt / cnt.sum(1, keepdims=True)
se = np.sqrt(P626 * (1 - P626) / cnt.sum(1, keepdims=True))
assert np.all(np.abs(Phat - P626) <= 4 * se + 1e-12)
print("estimated P from the simulated panel:\n", Phat)
# invariant distribution and ergodicity
evals, evecs = np.linalg.eig(P626.T)
assert np.sum(np.isclose(evals, 1)) == 1
pinf = np.real(evecs[:, np.isclose(evals, 1)][:, 0]); pinf /= pinf.sum()
assert np.allclose(pinf, [0, 0, 1])

theta_bar = 3.9796, eps_bar = 4.1837
P =
[[0.762  0.1804 0.0576]
 [0.     0.82   0.18  ]
 [0.     0.     1.    ]]


estimated P from the simulated panel:
 [[0.761096 0.181264 0.057641]
 [0.       0.821806 0.178194]
 [0.       0.       1.      ]]


## Exercise 6.27 --- Neal model with unemployment

$v(\theta,\epsilon)=\max\{\theta+\epsilon+\beta v(\theta,\epsilon),\ c+\beta\int v(\theta,\epsilon')dG,\ c+\beta\iint v\,dF\,dG\}$.
Same grids, $\beta=.95$, $c=1$. For $\theta\ge\bar\theta$ the job cutoff solves the McCall equation with offers
$\theta+\epsilon'$: $\theta+\bar\epsilon(\theta)-c=\frac{\beta}{1-\beta}\int_{\bar\epsilon(\theta)}(\epsilon'-\bar\epsilon(\theta))dG$, so $\bar\epsilon(\theta)$ falls with $\theta$.
Checks: VFI regions vs. these cutoffs; for $\theta<\bar\theta$ the stay/new-life boundary $(\theta+\epsilon)/(1-\beta)=Q$.

In [28]:
c627 = 1.
v627, reg627, C627, Q627 = neal(c=c627)
ib = np.argmax(C627 >= Q627 - 1e-9); thbar627 = th[ib]
assert np.all(C627[ib:] >= Q627 - 1e-9) and np.all(C627[:ib] < Q627)
epsbar627 = np.array([resw(c627 - t, b626, ep, q) for t in th[ib:]])
for i, t in enumerate(th[ib:]):
    assert np.array_equal(reg627[ib + i] == 3, ep >= epsbar627[i] - 1e-9) and np.all(reg627[ib + i] != 1)
for i in range(ib):
    assert np.array_equal(reg627[i] == 3, (th[i] + ep) / (1 - b626) >= Q627 - 1e-9) and np.all(reg627[i] != 2)
assert np.all(np.diff(epsbar627) <= 1e-12) and epsbar627[0] > epsbar627[-1]
print(f"theta_bar = {thbar627:.4f}; eps_bar(theta) from {epsbar627[0]:.4f} (theta={th[ib]:.3f}) to {epsbar627[-1]:.4f} (theta=5)")
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.0))
from matplotlib.colors import ListedColormap
cm = ListedColormap(["white", "0.6", "0.1"])
for a_, reg, ttl in ((ax[0], reg626, "6.26: Neal (no unemployment)"), (ax[1], reg627, f"6.27: with unemployment, $c={c627:g}$")):
    a_.pcolormesh(th, ep, reg.T, cmap=cm, vmin=1, vmax=3, shading="nearest", edgecolors="face", linewidth=0, rasterized=True)
    a_.set_xlabel(r"career $\theta$"); a_.set_ylabel(r"job $\epsilon$"); a_.set_title(ttl)
    a_.text(1.2, 2.3, "new life", fontsize=8); a_.text(4.0, 1.0, "new\njob", fontsize=8, color="white")
fig.tight_layout(); save_pdf(fig, "neal"); plt.close(fig)

theta_bar = 3.6735; eps_bar(theta) from 3.2645 (theta=3.673) to 3.0985 (theta=5)
